# BENCHMARK THUC NGHIEM CEGR: META-WORLD CONTINUOUS ROBOTIC MANIPULATION (MUC 2 - KHONG GIAN LIEN TUC)

**Thực nghiệm chuẩn hóa CEGR:** So sánh toàn diện 4 phương pháp chính (`EUREKA`, `CARD`, `CARD_REFL`, `CEGR`) cùng 6 biến thể Ablation (`CEGR_noPool`, `CEGR_noAnchors`, `CEGR_noClass`, `CEGR_noG2`, `CEGR_noG4`, `CEGR_noG5`) trên miền điều khiển cánh tay robot liên tục **Meta-World** (`reach-v2`, `door-open-v2`, `pick-place-v2`).

- **Tối ưu hóa mô-đun:** Sử dụng trực tiếp gói thư viện chuẩn `cegr` ([cegr/](file:///d:/CEGR/Cegr-official/cegr)) thay vì định nghĩa lại mã nguồn thủ công.
- **Trọn vẹn quy trình 3 pha:** Pha 1 (Nạp / chuẩn bị dataset mutants liên tục + natural rewards từ `manifest_metaworld.json`), Pha 2 (Vòng lặp sửa đổi + GateKeeper 5 cổng liên tục + Continuous Trajectory Attacker + Continuous RL Probe), Pha 3 (Bootstrap thống kê, biểu đồ và 4 bảng LaTeX).


In [ ]:
# =====================================================================
# CELL 0: CLONE KHO MÃ NGUỒN CEGR (KAGGLE / GOOGLE COLAB SETUP)
# =====================================================================
import os
import sys

# 1. Tự động clone kho mã nguồn CEGR nếu chạy trên Colab / Kaggle và chưa có mã nguồn
if not os.path.exists("cegr") and not os.path.exists("../cegr"):
    if not os.path.exists("CEGR"):
        print(">> Đang clone kho mã nguồn CEGR từ GitHub...")
        !git clone https://github.com/ToanThinh1208/CEGR.git
    else:
        print(">> Thư mục CEGR đã tồn tại trên đĩa.")
        # Bỏ comment nếu muốn tự động kéo code mới nhất từ GitHub:
        # !git -C CEGR pull
    
    if os.path.exists("CEGR"):
        %cd CEGR
elif os.path.exists("CEGR") and not os.path.exists("cegr"):
    %cd CEGR

# 2. Đảm bảo thư mục gốc repo nằm trong sys.path để import cegr
repo_root = os.path.abspath(".") if os.path.exists("cegr") else (os.path.abspath("..") if os.path.exists("../cegr") else os.path.abspath("CEGR"))
for p in [repo_root, os.path.abspath("."), os.path.abspath("..")]:
    if os.path.exists(os.path.join(p, "cegr")) and p not in sys.path:
        sys.path.insert(0, p)

# 3. Thông báo kiểm tra môi trường
print(f"✅ Thư mục làm việc hiện tại: {os.getcwd()}")
print(f"✅ Repo root trong sys.path: {repo_root}")

try:
    import cegr
    print(f"✅ Đã sẵn sàng nạp package 'cegr' tại: {cegr.__file__}")
except ImportError as e:
    print(f"⚠️ Lưu ý: Chưa thể nạp package cegr ({e}). Hãy đảm bảo bạn đang ở thư mục repo CEGR.")


In [ ]:
# =====================================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN & CẤU HÌNH TOÀN CỤC (+ cấu hình CEGR)
# =====================================================================
import subprocess, sys, os

# Gỡ các gói gây xung đột dependency trên Kaggle trước khi cài Torch + vLLM
!pip uninstall -y torchvision torchaudio cudf-cu12 cuml-cu12 dask-cuda pylibcudf-cu12 > /dev/null 2>&1
!pip install -q torch==2.6.0+cu124 --index-url https://download.pytorch.org/whl/cu124
!pip install -q vllm transformers accelerate tiktoken autoawq hf_transfer
# Cài đặt metaworld (tùy chọn nếu chạy với MuJoCo đầy đủ trên Colab/Kaggle)
# !pip install -q git+https://github.com/Farama-Foundation/Metaworld.git@master

# Đảm bảo đường dẫn import cegr package hoạt động cả khi chạy từ ./notebooks hoặc root
for p in [os.path.abspath('..'), os.path.abspath('.')]:
    if p not in sys.path: sys.path.insert(0, p)

import json, time, random, collections, hashlib, math, re, ast, builtins, itertools
import signal, threading
from dataclasses import dataclass, field, asdict
from typing import Dict, List, NamedTuple, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mp

# Lá cờ chọn môi trường thực thi: True = Colab, False = Kaggle
IS_COLAB = False   # True: Colab | False: Kaggle (Người dùng tự điền True hoặc False)

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 300,
    "axes.grid": True, "grid.alpha": 0.25,
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10
})

# ============== CẤU HÌNH THỰC NGHIỆM CHUẨN HÓA =============
MODEL_ID         = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"
MAX_MODEL_LEN    = 6144

FAST_SMOKE_TEST   = False

# Cấu hình Sinh tập Dataset (Pha 1) - Đồng bộ 100% điều kiện nghiên cứu với DoorKey
N_NATURAL_PROMPTS = 30  if FAST_SMOKE_TEST else 360 # Sinh 360 mẫu bằng Batch vLLM để lọc lấy >= 18-25 mẫu natural hacked
FORCE_REBUILD_DATASET = False # Đặt True nếu muốn ép buộc LLM chạy lại batch 360 prompts thay vì đọc cache

# ============== [CEGR] CẤU HÌNH THỰC NGHIỆM LIÊN TỤC META-WORLD ==============
ENV_NAME         = "door-open-v2"   # "reach-v2" | "door-open-v2" | "pick-place-v2"
HORIZON          = 150              # Horizon điều khiển liên tục (150 bước)
USE_MOCK         = True             # True: Mock physics chuẩn hóa | False: Metaworld MuJoCo engine
N_ROUNDS         = 4                # Ngân sách gọi Coder / job (CÙNG ngân sách 4 cuộc gọi cho mọi phương pháp)
N_SAMPLES        = 1 if FAST_SMOKE_TEST else 3   # seed LLM độc lập cho mỗi (reward gốc, phương pháp)
GEN_TEMP         = 0.95
REPAIR_TEMP      = 0.35
REPAIR_MAXTOK    = 900
EUREKA_POP       = 2                # số candidate / vòng của baseline Eureka-style

METHODS          = ["EUREKA", "CARD", "CARD_REFL", "CEGR"]
ABLATIONS        = ["CEGR_noPool", "CEGR_noAnchors", "CEGR_noClass", "CEGR_noG2", "CEGR_noG4", "CEGR_noG5"]
CONDS            = METHODS + ABLATIONS
N_MAIN_EVAL      = 3  if FAST_SMOKE_TEST else 3    # số reward gốc liên tục chạy đủ 4 phương pháp
N_ABL_EVAL       = 1  if FAST_SMOKE_TEST else 3    # số reward gốc chạy thêm 6 ablation

# --- GateKeeper / Continuous CEM Attacker / Probe ---
EPS_SIGNAL_REL   = 0.02             # eps tương đối so với |return quỹ đạo thành công tham chiếu|
EPS_SIGNAL_ABS   = 1e-3
RHO_THR          = 0.70             # ngưỡng Spearman ở Gate 4
M_CE_PER_ROUND   = 2                # tối đa CE thêm vào pool từ mỗi nguồn / vòng
POOL_MAX         = 40
STALL_K          = 2
MAX_INVARIANTS   = 8
ATTACKER_MODE    = "continuous_cem" # Continuous Cross-Entropy Method Attacker
CE_STREAM_SEED, SIGNAL_STREAM_SEED, EVAL_STREAM_SEED = 1000, 2000, 3000   # 3 luồng seed TÁCH BIỆT

SEED             = 42
SUFFIX           = f"__{ENV_NAME.replace('-', '_')}__continuous"

# ============== QUẢN LÝ ĐƯỜNG DẪN & HẬU TỐ FILE OUTPUT ==============
ON_COLAB  = IS_COLAB
ON_KAGGLE = not IS_COLAB

if IS_COLAB:
    OUT_DIR    = "/content/cegr_out"
    LEGACY_DIR = "/content/cegr_out"
else:
    OUT_DIR    = "/kaggle/working/cegr"
    LEGACY_DIR = "/kaggle/input/cegr-v1-results" if os.path.exists("/kaggle/input/cegr-v1-results") else OUT_DIR

os.makedirs(OUT_DIR, exist_ok=True)

def out_path(base_name: str, ext: str, use_suffix: bool = True) -> str:
    """Sinh đường dẫn lưu file trong OUT_DIR có gắn hậu tố SUFFIX."""
    sfx = SUFFIX if use_suffix else ""
    return os.path.join(OUT_DIR, f"{base_name}{sfx}{ext}")

def find_legacy_file(filename: str) -> str:
    """Tìm file gốc ở OUT_DIR hoặc LEGACY_DIR nếu có."""
    p_out = os.path.join(OUT_DIR, filename)
    if os.path.exists(p_out):
        return p_out
    p_leg = os.path.join(LEGACY_DIR, filename)
    if os.path.exists(p_leg):
        return p_leg
    return p_out

random.seed(SEED); np.random.seed(SEED)
print(f"✓ Cấu hình hoàn tất | Mode = {SUFFIX}")
print(f"  -> Task: {ENV_NAME}, Horizon: {HORIZON}, Mock Physics: {USE_MOCK}")
print(f"  -> Methods ({len(METHODS)}): {METHODS}")
print(f"  -> Ablations ({len(ABLATIONS)}): {ABLATIONS}")
print(f"  -> OUT_DIR = {OUT_DIR}")


# CELL 2: MÔI TRƯỜNG META-WORLD, SEMANTIC STATE WRAPPER & CODE SANDBOX (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 2: KHỞI TẠO MÔI TRƯỜNG META-WORLD & CODE SANDBOX TỪ CEGR
# =====================================================================
from cegr.envs.metaworld_env import make_metaworld_env, MetaWorldEnvWrapper
from cegr.core.code_sandbox import (
    RewardError, check_code, code_fingerprint, ast_node_count,
    make_namespace
)
from cegr.envs.base_env import Step, Trajectory

env = make_metaworld_env(ENV_NAME, horizon=HORIZON, use_mock=USE_MOCK)
obs, info = env.reset()
print(f"✓ Môi trường Meta-World '{ENV_NAME}' sẵn sàng: horizon={env.horizon}, use_mock={env.use_mock}")
print("✓ Semantic observation dictionary:")
for k, v in obs.items():
    if k != "raw_obs":
        print(f"    {k}: {v}")


# CELL 3: CONTINUOUS TRAJECTORY ATTACKER, CONTINUOUS RL PROBE & VERIFIED FIX (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 3: CONTINUOUS TRAJECTORY ATTACKER & CONTINUOUS RL PROBE TỪ CEGR
# =====================================================================
from cegr.solvers.trajectory_attacker import ContinuousTrajectoryAttacker
from cegr.solvers.continuous_rl_probe import ContinuousRLProbe
from cegr.solvers.verified_fix import evaluate_verified_fix

attacker = ContinuousTrajectoryAttacker(env, cem_iterations=10, population_size=50)
# Chỉ truyền tham số env, lược bỏ hoàn toàn các keyword arguments không được hỗ trợ
rl_probe = ContinuousRLProbe(env)

print("✓ Continuous Trajectory Attacker (CEM) sẵn sàng: iters=10, pop=50")
print("✓ Continuous RL Probe (SAC/PPO) đã được khởi tạo thành công")
print("✓ Verified Fix Evaluator sẵn sàng cho miền liên tục Meta-World")

# CELL 4: QUỸ ĐẠO THAM CHIẾU (T0-T3), COUNTEREXAMPLE POOL & DYNAMIC ANCHORS (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 4: QUỸ ĐẠO THAM CHIẾU, COUNTEREXAMPLE POOL & DYNAMIC ANCHORS
# =====================================================================
from cegr.synthesis.ce_pool import CEPool, make_ce, Counterexample
from cegr.synthesis.dynamic_anchors import DynamicAnchorsManager
from cegr.synthesis.signal_checker import signal_check, SignalChecker
from cegr.synthesis.static_antihack import static_checks
from cegr.synthesis.runner import Kit, build_metaworld_kit

# Khởi tạo Kit tham chiếu độc lập cho Meta-World (3 luồng seed tách biệt)
kit = build_metaworld_kit(env, ce_seed=CE_STREAM_SEED, signal_seed=SIGNAL_STREAM_SEED, eval_seed=EVAL_STREAM_SEED)
print("✓ Khởi tạo Kit tham chiếu Meta-World hoàn tất:")
print(f"  -> {len(kit.p0)} cặp Counterexample P0 chuẩn")
print(f"  -> {len(kit.refs)} quỹ đạo tham chiếu cốt lõi: {list(kit.refs.keys())}")
print(f"  -> {len(kit.signal_trajs)} quỹ đạo Signal Stream (Seed={SIGNAL_STREAM_SEED})")
print(f"  -> {len(kit.eval_trajs)} quỹ đạo Eval Stream (Seed={EVAL_STREAM_SEED})")

anchors_mgr = DynamicAnchorsManager(domain="metaworld")
print(f"✓ Dynamic Anchors Manager khởi tạo cho miền 'metaworld': {list(anchors_mgr.anchors.keys())}")


# CELL 5: GATEKEEPER 5 CỔNG CHO MIỀN LIÊN TỤC & CÁC RUNNER PHƯƠNG PHÁP (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 5: GATEKEEPER 5 CỔNG & CÁC RUNNER PHƯƠNG PHÁP
# =====================================================================
from cegr.synthesis.gatekeeper import GateKeeper, run_gatekeeper, build_feedback
from cegr.synthesis.assimilator import extract_experience, InvariantAssimilator
from cegr.synthesis.runner import CegrRunner
from cegr.baselines.eureka.runner import EurekaRunner
from cegr.baselines.card.runner import CardRunner
from cegr.baselines.orig_runner import OrigRunner

gk = GateKeeper(env, kit)
print("✓ GateKeeper 5 cổng (hỗ trợ Continuous Attacker & Continuous Probe) đã sẵn sàng.")
print("✓ Invariant Assimilator đã sẵn sàng.")
print("✓ Các Runner (CEGR, EurekaRunner, CardRunner, OrigRunner) đã được nạp sẵn sàng.")


# CELL 6: ĐỘNG CƠ vLLM BATCH INFERENCE (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 6: ĐỘNG CƠ vLLM BATCH INFERENCE
# =====================================================================
from cegr.engine.llm_vllm import init_llm, llm_batch_chat, extract_code

try:
    llm_inst = init_llm(model_id=MODEL_ID, max_model_len=MAX_MODEL_LEN)
    print(f"✓ Động cơ vLLM khởi tạo thành công: {MODEL_ID} (max_model_len={MAX_MODEL_LEN})")
except Exception as e:
    print(f"⚠️ Lưu ý: Chưa khởi tạo được vLLM trực tiếp ({e}). Sẽ sử dụng fallback inference khi cần.")
    llm_inst = None


# CELL 7: PHA 1 (BUILD DATASET) - 6 HỌ MUTANTS & NATURAL REWARDS META-WORLD (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 7: PHA 1 (BUILD DATASET) - 6 HỌ MUTANTS & NATURAL REWARDS META-WORLD
# =====================================================================
from cegr.synthesis.dataset import (
    build_calibrated_mutants, generate_natural_rewards,
    MUTANT_FAMILIES_METAWORLD, NATURAL_PERSONAS_METAWORLD
)

base_manifest_path = os.path.join(OUT_DIR, f"manifest_{ENV_NAME.replace('-', '_')}.json")

if os.path.exists(base_manifest_path) and not FORCE_REBUILD_DATASET:
    with open(base_manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    print(f"✓ Nạp manifest có sẵn: {len(manifest)} rewards từ {base_manifest_path}")
else:
    print(f"--- Đang sinh dataset chuẩn hóa cho môi trường Meta-World {ENV_NAME} (Đồng bộ điều kiện với DoorKey) ---")
    mut_list = build_calibrated_mutants(env)
    print(f"  -> Đã chuẩn bị {len(mut_list)} calibrated continuous mutants qua 6 họ.")
    
    try:
        print(f"  -> Đang gửi {N_NATURAL_PROMPTS} prompts vào vLLM batch chat để sinh natural rewards...")
        nat_list, gen_stats = generate_natural_rewards(
            env, n_prompts=N_NATURAL_PROMPTS, seed=SEED
        )
        print(f"  -> Đã sinh {len(nat_list)} natural hacked continuous rewards. Thống kê: {gen_stats}")
    except Exception as e:
        print(f"  -> Lưu ý: Không khởi tạo được vLLM batch chat ({e}), sử dụng tập starter mutants.")
        nat_list = []
    
    manifest = mut_list + nat_list
    with open(base_manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    print(f"✓ Đã lưu dataset gồm {len(manifest)} rewards tại {base_manifest_path}")

print(f"Danh sách {len(manifest)} rewards sẵn sàng thực nghiệm:")
for idx, r in enumerate(manifest):
    print(f"  [{idx+1}] ID: {r.get('id')}, Task: {r.get('task')}, Origin: {r.get('origin', 'mutant')}, Family: {r.get('family', 'unknown')}")


# CELL 8: PHA 2 - CHẠY THỰC NGHIỆM ĐỐI SÁNH CEGR, BASELINE & ABLATION TRÊN META-WORLD


In [ ]:
# =====================================================================
# CELL 8: PHA 2 - CHẠY THỰC NGHIỆM ĐỐI SÁNH CEGR, BASELINE & ABLATION
# =====================================================================
from cegr.synthesis.dataset import METAWORLD_TASK_DESCS, get_metaworld_api_desc
METAWORLD_TASK_DESC = METAWORLD_TASK_DESCS.get(ENV_NAME, f"Task: Robotic continuous manipulation ({ENV_NAME}).")
METAWORLD_API_DESC = get_metaworld_api_desc(ENV_NAME)

results_file = out_path("results_main_metaworld", ".jsonl")
all_results = []

# Tải checkpoint kết quả đã có nếu chạy nối tiếp
if os.path.exists(results_file):
    with open(results_file, "r", encoding="utf-8") as f:
        all_results = [json.loads(line) for line in f if line.strip()]
    print(f"✓ Nạp checkpoint kết quả: {len(all_results)} jobs đã hoàn tất.")

# Cấu hình runner đồng nhất ngân sách 4 LLM calls
runner_config = {
    "max_rounds": N_ROUNDS,
    "llm": {"temperature_repair": REPAIR_TEMP, "max_tokens": REPAIR_MAXTOK},
    "scoring_profile": "continuous_metaworld",
    "domain": "metaworld",
}

runners = {
    "ORIG": OrigRunner(env, runner_config),
    "EUREKA": EurekaRunner(env, runner_config, task_desc=METAWORLD_TASK_DESC, api_desc=METAWORLD_API_DESC),
    "CARD": CardRunner(env, runner_config, use_reflection_memory=False, task_desc=METAWORLD_TASK_DESC, api_desc=METAWORLD_API_DESC),
    "CARD_REFL": CardRunner(env, runner_config, use_reflection_memory=True, task_desc=METAWORLD_TASK_DESC, api_desc=METAWORLD_API_DESC),
    "CEGR": CegrRunner(env, runner_config, method_name="CEGR", task_desc=METAWORLD_TASK_DESC, api_desc=METAWORLD_API_DESC),
    "CEGR_noPool": CegrRunner(env, runner_config, method_name="CEGR_noPool", variant_cfg={"pool": False}, task_desc=METAWORLD_TASK_DESC, api_desc=METAWORLD_API_DESC),
    "CEGR_noAnchors": CegrRunner(env, runner_config, method_name="CEGR_noAnchors", variant_cfg={"anchors": "none"}, task_desc=METAWORLD_TASK_DESC, api_desc=METAWORLD_API_DESC),
    "CEGR_noClass": CegrRunner(env, runner_config, method_name="CEGR_noClass", variant_cfg={"classify": False}, task_desc=METAWORLD_TASK_DESC, api_desc=METAWORLD_API_DESC),
    "CEGR_noG2": CegrRunner(env, runner_config, method_name="CEGR_noG2", variant_cfg={"gates": [1, 3, 4, 5]}, task_desc=METAWORLD_TASK_DESC, api_desc=METAWORLD_API_DESC),
    "CEGR_noG4": CegrRunner(env, runner_config, method_name="CEGR_noG4", variant_cfg={"gates": [1, 2, 3, 5]}, task_desc=METAWORLD_TASK_DESC, api_desc=METAWORLD_API_DESC),
    "CEGR_noG5": CegrRunner(env, runner_config, method_name="CEGR_noG5", variant_cfg={"gates": [1, 2, 3, 4]}, task_desc=METAWORLD_TASK_DESC, api_desc=METAWORLD_API_DESC),
}

# 1. Chạy 4 phương pháp chính
eval_records = manifest[:N_MAIN_EVAL]
for cond in METHODS:
    already_run = [r for r in all_results if r.get("method") == cond]
    if len(already_run) >= len(eval_records) * N_SAMPLES:
        print(f"✓ Bỏ qua {cond}: đã có đủ {len(already_run)} kết quả.")
        continue
    print(f"\n>>> Đang chạy phương pháp: {cond} trên {len(eval_records)} rewards...")
    runner = runners[cond]
    cond_res = runner.run_wave_batch(eval_records, n_samples=N_SAMPLES, label=cond) if hasattr(runner, "run_wave_batch") else runner.run_batch(eval_records, n_samples=N_SAMPLES)
    all_results.extend(cond_res)
    with open(results_file, "w", encoding="utf-8") as f:
        for item in all_results:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"✓ Hoàn thành {cond}: {len(cond_res)} jobs ghi nhận.")

# 2. Chạy 6 biến thể Ablation
abl_records = manifest[:N_ABL_EVAL]
for abl in ABLATIONS:
    already_run = [r for r in all_results if r.get("method") == abl]
    if len(already_run) >= len(abl_records) * N_SAMPLES:
        print(f"✓ Bỏ qua {abl}: đã có đủ {len(already_run)} kết quả.")
        continue
    print(f"\n>>> Đang chạy biến thể: {abl} trên {len(abl_records)} rewards...")
    runner = runners[abl]
    abl_res = runner.run_wave_batch(abl_records, n_samples=N_SAMPLES, label=abl) if hasattr(runner, "run_wave_batch") else runner.run_batch(abl_records, n_samples=N_SAMPLES)
    all_results.extend(abl_res)
    with open(results_file, "w", encoding="utf-8") as f:
        for item in all_results:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"✓ Hoàn thành {abl}: {len(abl_res)} jobs ghi nhận.")

print(f"\n✓ Hoàn tất Pha 2: Tổng cộng {len(all_results)} kết quả thực nghiệm được lưu tại {results_file}.")


# CELL 9: PHA 3 - BOOTSTRAP THỐNG KÊ, BIỂU ĐỒ & BẢNG LATEX (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 9: PHA 3 - BOOTSTRAP THỐNG KÊ, BIỂU ĐỒ & XUẤT BẢNG LATEX
# =====================================================================
from cegr.eval.bootstrap import bootstrap_ci, per_reward, paired_test
from cegr.eval.report_generator import generate_latex_tables, plot_evaluation_figures

df = pd.DataFrame(all_results)
if not df.empty:
    print("Thống kê mẫu kết quả theo method trên Meta-World:")
    cols_show = [c for c in ["vf", "tier1_safe", "loose_dp", "llm_calls", "rl_episodes"] if c in df.columns]
    if cols_show:
        print(df.groupby("method")[cols_show].mean())
    else:
        print(df.groupby("method").size())

    # Vẽ biểu đồ kết quả (300 DPI)
    plot_evaluation_figures(df, out_dir=OUT_DIR)
    
    # Xuất các bảng LaTeX chuẩn ICLR/IEEE
    out_tex_path = out_path("tables_report_metaworld", ".tex")
    generate_latex_tables(df, out_tex_path=out_tex_path)
    print(f"✓ Đã lưu biểu đồ và các bảng LaTeX tại {OUT_DIR}.")
else:
    print("Chưa có dữ liệu kết quả để xuất biểu đồ.")


# CELL 10: TEST TOÀN VẸN VÀ NGHIỆM THU TỰ ĐỘNG TRÊN META-WORLD


In [ ]:
# =====================================================================
# CELL 10: TEST TOÀN VẸN VÀ NGHIỆM THU TỰ ĐỘNG TRÊN META-WORLD
# =====================================================================
print(f">>> Bắt đầu chạy test nghiệm thu trên môi trường Meta-World {ENV_NAME}...")
assert env is not None, "Env chưa khởi tạo"
assert kit is not None and len(kit.p0) >= 4, "Kit P0 Meta-World chưa khởi tạo đúng"
assert len(runners) >= 10, "Số lượng runners không đầy đủ"

# Test thử một reward hàm cơ bản trên sandbox liên tục
test_code = (
    "def reward_components(s, a, s2):\n"
    "    c = {}\n"
    "    c['step'] = -0.01\n"
    "    d = float(np.linalg.norm(np.array(s2['hand_pos']) - np.array(s2['obj_pos'])))\n"
    "    c['reach'] = -1.0 * d\n"
    "    c['goal'] = 10.0 if s2.get('in_target', False) else 0.0\n"
    "    return c\n"
)
tree = check_code(test_code)
assert tree is not None, "Sandbox AST check failed"

ns = make_namespace(env)
exec(compile(test_code, "<test_mw>", "exec"), ns)
fn = ns["reward_components"]

s, _ = env.reset()
a = np.array([0.0, 0.1, -0.05, 0.0], dtype=np.float32)
s2, _, _, _, _ = env.step(a)
out_r = fn(s, a, s2)
assert isinstance(out_r, dict) and "reach" in out_r, "Hàm reward không trả về đúng định dạng dict"
print(f"✓ Test mẫu hàm reward liên tục thành công: {out_r}")

# Test GateKeeper trên test_code
from cegr.synthesis.ce_pool import CEPool
test_pool = CEPool(max_size=40)
gk_res = run_gatekeeper(test_code, {"gates": [1, 2, 3]}, test_pool, rnd=0, env=env, kit=kit, gk=gk)
print(f"✓ Test GateKeeper Gate 1-3: passed={gk_res['passed']}, gates_passed={gk_res['gates_passed']}")
print(f"✓ Toàn bộ kiểm thử nghiệm thu trên Meta-World {ENV_NAME} đã hoàn thành tốt đẹp!")
